## verifier_incoherence_5465_8347

**Fichier(s) source :** `./data/bofip_stock_live_20260521.tgz` (stock du 21.05.2026)

**Fichier(s) de sortie :** Aucun (résultats affichés en sortie)

**Description :** Vérification d'une incohérence bidirectionnelle de renvois entre 5465-PGP (formulaire 6493-N) et 8347-PGP : contrôle si 5465-PGP se déclare requis par 8347-PGP (isRequiredBy) et si 8347-PGP déclare requérir 5465-PGP en retour.

In [1]:
import tarfile, os
import xml.etree.ElementTree as ET

ARCHIVE = r"./data/bofip_stock_live_20260521.tgz"
IDS = {"5465-PGP", "8347-PGP"}

resultats = {}

with tarfile.open(ARCHIVE, "r:gz") as tar:
    for m in tar.getmembers():
        if not m.isfile():
            continue
        chemin = m.name.replace("\\", "/")
        if not chemin.endswith("document.xml"):
            continue
        for code in IDS:
            if "/" + code + "/" in chemin:
                f = tar.extractfile(m)
                if f is None:
                    continue
                data = f.read()
                root = ET.fromstring(data)
                
                # Extraire titre et toutes les relations
                titre = ""
                relations = []
                for el in root.iter():
                    tag = el.tag.rsplit("}", 1)[-1] if "}" in el.tag else el.tag
                    if tag == "title" and el.text:
                        titre = el.text.strip()
                    if tag == "relation" and el.text:
                        rel_type = ""
                        for k, v in el.attrib.items():
                            if "type" in k.lower():
                                rel_type = v
                        relations.append((rel_type, el.text.strip()))
                
                resultats[code] = {"titre": titre, "relations": relations}
                print(f"Trouvé : {code} — titre : {titre}")
                break

print("\n" + "=" * 60)
print("VÉRIFICATION DU GRAPHE BIDIRECTIONNEL")
print("=" * 60)

# 5465-PGP déclare-t-il isRequiredBy 8347-PGP ?
if "5465-PGP" in resultats:
    print(f"\n5465-PGP ({resultats['5465-PGP']['titre']}) :")
    print(f"  Relations ({len(resultats['5465-PGP']['relations'])}) :")
    for rel_type, cible in resultats["5465-PGP"]["relations"]:
        marqueur = " ← TROUVÉ" if "8347" in cible else ""
        print(f"    {rel_type:20s} → {cible}{marqueur}")
    
    cite_8347 = any("8347" in c for _, c in resultats["5465-PGP"]["relations"])
    print(f"\n  5465-PGP cite 8347-PGP ? {'OUI' if cite_8347 else 'NON'}")

# 8347-PGP déclare-t-il requires 5465-PGP ?
if "8347-PGP" in resultats:
    print(f"\n8347-PGP ({resultats['8347-PGP']['titre']}) :")
    print(f"  Relations ({len(resultats['8347-PGP']['relations'])}) :")
    for rel_type, cible in resultats["8347-PGP"]["relations"]:
        marqueur = " ← TROUVÉ" if "5465" in cible else ""
        print(f"    {rel_type:20s} → {cible}{marqueur}")
    
    cite_5465 = any("5465" in c for _, c in resultats["8347-PGP"]["relations"])
    print(f"\n  8347-PGP cite 5465-PGP ? {'OUI' if cite_5465 else 'NON'}")

print("\n" + "=" * 60)
if "5465-PGP" in resultats and "8347-PGP" in resultats:
    if cite_8347 and not cite_5465:
        print("CONCLUSION : incohérence confirmée.")
        print("5465-PGP se déclare requis par 8347-PGP,")
        print("mais 8347-PGP ne déclare pas requérir 5465-PGP.")
        print("Le lien n'existe que dans un sens.")
    elif cite_8347 and cite_5465:
        print("CONCLUSION : pas d'incohérence. Le lien est bidirectionnel.")
    else:
        print("CONCLUSION : résultat inattendu, vérifier manuellement.")
print("=" * 60)

Trouvé : 8347-PGP — titre : CAD - Mise à jour du plan - Confection des documents d'arpentage - Travaux et signature
Trouvé : 5465-PGP — titre : CAD - Version PDF du procès-verbal n° 6493 N

VÉRIFICATION DU GRAPHE BIDIRECTIONNEL

5465-PGP (CAD - Version PDF du procès-verbal n° 6493 N) :
  Relations (7) :
    isRequiredBy         → Contenu:5235-PGP
    isRequiredBy         → Contenu:5258-PGP
    isRequiredBy         → Contenu:5176-PGP
    isRequiredBy         → Contenu:5184-PGP
    isRequiredBy         → Contenu:8346-PGP
    isRequiredBy         → Contenu:5190-PGP
    isRequiredBy         → Contenu:8347-PGP ← TROUVÉ

  5465-PGP cite 8347-PGP ? OUI

8347-PGP (CAD - Mise à jour du plan - Confection des documents d'arpentage - Travaux et signature) :
  Relations (15) :
    references           → Actualite:12500-PGP
    requires             → Fichier:13093-PGP
    references           → Contenu:13690-PGP
    references           → Contenu:13691-PGP
    requires             → Fichier:13694-PG